# Part A :- Concept Appliction 

## Import Libraries

In [18]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

## Load and preprocess dataset

In [19]:
def load_and_prepare_data(test_size=0.2, random_state=42):
    """Load the digits dataset, split it into training and testing sets, and scale the features."""
    digits = load_digits()
    X = digits.data
    y = digits.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test


## Train SVM with GridSearchCV

In [20]:
def train_svm(X_train, y_train):
    """ Train an SVM classifier using GridSearchCV to find the best hyperparameters."""
    param_grid = {
        'C': [1, 10, 100],
        'gamma': [0.001, 0.01, 0.1]
    }

    grid = GridSearchCV(
        SVC(kernel='rbf'),
        param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print("Best SVM Parameters:", grid.best_params_)
    return grid.best_estimator_

## Train KNN

In [21]:
def train_knn(X_train, y_train, k=3):
    """ Train a KNN classifier with the specified number of neighbors."""
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    return knn

## Evaluate model

In [22]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n{model_name} Accuracy: {acc * 100:.4f} \n")
    print(f"{model_name} Confusion Matrix:\n{cm}")
    print(f"{model_name} Classification Report:\n")
    print(classification_report(y_test, y_pred))

    return y_pred, cm


## Find most confused digit pairs

In [23]:
def find_confused_pairs(cm):
    cm_copy = cm.copy()
    np.fill_diagonal(cm_copy, 0)

    max_confusion = np.max(cm_copy)
    pairs = np.argwhere(cm_copy == max_confusion)

    print("\nMost confused digit pairs:")
    for pair in pairs:
        print(f"{pair[0]} vs {pair[1]} ({max_confusion} mistakes)")

In [26]:
X_train, X_test, y_train, y_test = load_and_prepare_data()

In [27]:
knn_model = train_knn(X_train, y_train)

knn_pred, knn_cm = evaluate_model(knn_model, X_test, y_test, "KNN")

find_confused_pairs(knn_cm)


KNN Accuracy: 96.6667 

KNN Confusion Matrix:
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 35  1  0  0  0  0  0  0  0]
 [ 0  0 35  0  0  0  0  0  0  0]
 [ 0  0  1 36  0  0  0  0  0  0]
 [ 0  0  0  0 34  0  0  2  0  0]
 [ 0  0  0  0  0 37  0  0  0  0]
 [ 0  0  0  0  0  0 36  0  0  0]
 [ 0  0  1  0  0  0  0 35  0  0]
 [ 0  3  1  0  0  0  0  0 31  0]
 [ 0  0  0  0  1  0  1  0  1 33]]
KNN Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       0.92      0.97      0.95        36
           2       0.90      1.00      0.95        35
           3       1.00      0.97      0.99        37
           4       0.97      0.94      0.96        36
           5       1.00      1.00      1.00        37
           6       0.97      1.00      0.99        36
           7       0.95      0.97      0.96        36
           8       0.97      0.89      0.93        35
           9       1.00      0.92      0.96        36

In [28]:
svm_model = train_svm(X_train, y_train)

svm_pred, svm_cm = evaluate_model(svm_model, X_test, y_test, "SVM")

find_confused_pairs(svm_cm)

Best SVM Parameters: {'C': 100, 'gamma': 0.01}

SVM Accuracy: 98.3333 

SVM Confusion Matrix:
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 35  0  0  1  0  0  0  0  0]
 [ 0  0 35  0  0  0  0  0  0  0]
 [ 0  0  0 37  0  0  0  0  0  0]
 [ 0  0  0  0 35  0  0  1  0  0]
 [ 0  0  0  0  0 37  0  0  0  0]
 [ 0  0  0  0  0  0 36  0  0  0]
 [ 0  0  0  0  0  0  0 36  0  0]
 [ 0  1  0  0  1  0  0  0 33  0]
 [ 0  0  0  0  0  0  1  1  0 34]]
SVM Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       0.97      0.97      0.97        36
           2       1.00      1.00      1.00        35
           3       1.00      1.00      1.00        37
           4       0.95      0.97      0.96        36
           5       1.00      1.00      1.00        37
           6       0.97      1.00      0.99        36
           7       0.95      1.00      0.97        36
           8       1.00      0.94      0.97        35
      